# 00. Разведка структуры БД для Look-Alike & Behavioral Segmentation

> **Краткое резюме для руководителя**: Этот ноутбук исследует структуру Hive-базы данных, чтобы найти таблицы и поля, необходимые для двух моделей: (1) Look-Alike — поиск проспектов, похожих на лучших привлечённых клиентов, (2) Поведенческая сегментация — кластеризация по паттернам транзакций. Результат — карта данных: какие таблицы содержат ОКВЭД, регион, выручку, модельный сегмент, статус клиента/проспекта.

**Что ищем**:

| Данные | Зачем | Статус |
|--------|-------|--------|
| ОКВЭД (отрасль) | Профиль проспекта для look-alike | ? |
| Регион | Профиль проспекта для look-alike | ? |
| Выручка | Размер компании, scoring | ? |
| Модельный сегмент | Пересечение с поведенческой сегментацией | ? |
| Статус (клиент / проспект) | Разделение на эталон vs целевую группу | ? |
| Транзакции (суммы, даты, направления) | Поведенческие фичи | Есть (`paymentcounteragent_stran`) |
| Контрагенты | Кол-во, динамика | Есть (`paymentcounteragent_stran`) |

---

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from IPython.display import display, HTML

In [ ]:
# =====================================================
# ПАРАМЕТРЫ
# =====================================================

HIVE_DATABASE = 's_dmrb'      # Основная БД (из существующего проекта)

# Дополнительные БД для поиска (CRM, справочники, модели)
# Добавьте сюда другие базы, если знаете их имена
EXTRA_DATABASES = [
    # 's_crm',
    # 's_mdl',
    # 's_ref',
]

In [ ]:
# На MDP SparkSession уже инициализирована платформой
try:
    _ = spark.sparkContext.appName
    print(f'SparkSession already available: {spark.sparkContext.appName}')
except Exception:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder \
        .appName('LookAlike_DB_Exploration') \
        .enableHiveSupport() \
        .getOrCreate()
    print('SparkSession created')

print(f'Spark version: {spark.version}')

---
## 1. Полный список таблиц в основной БД

Сначала получим полный перечень таблиц и найдём потенциально релевантные по именам.

In [ ]:
spark.sql(f'USE {HIVE_DATABASE}')
tables_df = spark.sql('SHOW TABLES').toPandas()
print(f'Всего таблиц в {HIVE_DATABASE}: {len(tables_df)}')
display(tables_df)

In [ ]:
# Поиск таблиц по ключевым словам
KEYWORDS = [
    'okved', 'okvad', 'nace',           # отрасль / ОКВЭД
    'region', 'address', 'addr', 'geo',  # регион / география
    'revenue', 'turnover', 'income',     # выручка
    'segment', 'model', 'rating',        # модельный сегмент
    'prospect', 'lead', 'crm',           # проспекты / CRM
    'status', 'client_status',           # статус клиента
    'taxpayer', 'inn', 'nalog',          # налоговые данные (СПАРК / ФНС)
    'company', 'org', 'jurid',           # компании / ЮЛ
    'industry', 'branch', 'sector',      # отрасль
    'sales', 'channel', 'manager',       # каналы продаж / менеджеры
    'product', 'deal', 'contract',       # продукты / сделки
    'spark', 'extern',                   # внешние данные
]

table_names = tables_df['tableName'].str.lower() if 'tableName' in tables_df.columns else tables_df.iloc[:, 1].str.lower()

print('Таблицы, потенциально содержащие нужные данные:')
print('=' * 70)
for kw in KEYWORDS:
    matches = table_names[table_names.str.contains(kw, case=False, na=False)]
    if not matches.empty:
        print(f'\n  [{kw}]:')
        for t in matches.values:
            print(f'    - {t}')

---
## 2. Поиск в дополнительных БД

Если есть доступ к другим БД (CRM, справочники, модели) — ищем там.

In [ ]:
# Попробуем найти доступные базы данных
try:
    all_dbs = spark.sql('SHOW DATABASES').toPandas()
    print(f'Доступные базы данных ({len(all_dbs)}):')
    display(all_dbs)
except Exception as e:
    print(f'Не удалось получить список БД: {e}')
    all_dbs = pd.DataFrame()

In [ ]:
# Поиск таблиц по ключевым словам в дополнительных БД
for db in EXTRA_DATABASES:
    print(f'\n{"=" * 60}')
    print(f'Database: {db}')
    print('=' * 60)
    try:
        extra_tables = spark.sql(f'SHOW TABLES IN {db}').toPandas()
        print(f'  Таблиц: {len(extra_tables)}')
        extra_names = extra_tables['tableName'].str.lower() if 'tableName' in extra_tables.columns else extra_tables.iloc[:, 1].str.lower()
        for kw in KEYWORDS:
            matches = extra_names[extra_names.str.contains(kw, case=False, na=False)]
            if not matches.empty:
                print(f'  [{kw}]: {list(matches.values)}')
    except Exception as e:
        print(f'  Ошибка доступа: {e}')

---
## 3. Детальный анализ таблицы клиентов (`client_sdim`)

Из предыдущего проекта известно: `client_sdim` содержит 182M строк, 65 полей.  
ОКВЭД и регион **НЕ найдены** (верифицировано 2026-02-21).  
Но проверим ВСЕ 65 колонок — возможно, есть поля с другими именами.

In [ ]:
def describe_table_full(table_name, database=HIVE_DATABASE):
    """Показать полную схему таблицы с типами данных."""
    full_name = f'{database}.{table_name}'
    print(f'\n{"=" * 70}')
    print(f'DESCRIBE {full_name}')
    print('=' * 70)
    try:
        desc = spark.sql(f'DESCRIBE {full_name}').toPandas()
        # Отфильтровываем partition info строки
        desc = desc[~desc['col_name'].str.startswith('#')]
        desc = desc[desc['col_name'].str.strip() != '']
        print(f'Колонок: {len(desc)}')
        display(desc)
        return desc
    except Exception as e:
        print(f'  ERROR: {e}')
        return pd.DataFrame()

In [ ]:
client_schema = describe_table_full('client_sdim')

In [ ]:
# Поиск колонок, которые МОГУТ содержать ОКВЭД, регион, сегмент, статус
COLUMN_KEYWORDS = [
    'okved', 'okvad', 'nace', 'activ', 'ind',    # отрасль
    'region', 'address', 'city', 'terr', 'geo',   # регион
    'segment', 'model', 'categ', 'class', 'group', # сегмент
    'status', 'state', 'stage',                     # статус
    'revenue', 'income', 'profit', 'turnover',      # выручка
    'prospect', 'lead', 'attract', 'acqui',         # проспект
    'sales', 'manager', 'channel',                  # менеджер / канал
    'rating', 'score', 'risk',                      # рейтинг / скоринг
]

if not client_schema.empty:
    col_names = client_schema['col_name'].str.lower()
    print('Потенциально релевантные колонки в client_sdim:')
    print('-' * 50)
    for kw in COLUMN_KEYWORDS:
        matches = client_schema[col_names.str.contains(kw, case=False, na=False)]
        if not matches.empty:
            for _, row in matches.iterrows():
                print(f'  [{kw:12s}]  {row["col_name"]:35s}  ({row["data_type"]})')

In [ ]:
# Просмотр примера данных client_sdim — ВСЕ колонки
print('Пример строк client_sdim (все колонки):')
spark.sql(f"""
    SELECT *
    FROM {HIVE_DATABASE}.client_sdim
    LIMIT 5
""").show(truncate=False, vertical=True)

---
## 4. Поиск таблиц с ОКВЭД

ОКВЭД может быть в:
- Справочнике видов деятельности
- Таблице ИНН / налоговой регистрации
- Внешних данных (СПАРК, ЕГРЮЛ)

In [ ]:
# Поиск по ВСЕМ таблицам БД: какие содержат колонку с 'okved' или 'nace'
tables_with_okved = []

for tbl in table_names.values:
    try:
        desc = spark.sql(f'DESCRIBE {HIVE_DATABASE}.{tbl}').toPandas()
        cols = desc['col_name'].str.lower()
        okved_cols = cols[cols.str.contains('okved|nace|activ', case=False, na=False)]
        if not okved_cols.empty:
            tables_with_okved.append((tbl, list(okved_cols.values)))
    except Exception:
        pass

print(f'Таблицы с колонками, содержащими "okved/nace/activ":')
if tables_with_okved:
    for tbl, cols in tables_with_okved:
        print(f'  {tbl}: {cols}')
else:
    print('  Не найдено в основной БД.')
    print('  -> Попробуйте добавить другие БД в EXTRA_DATABASES и перезапустить.')

---
## 5. Поиск таблиц с регионом / адресом

In [ ]:
tables_with_region = []

for tbl in table_names.values:
    try:
        desc = spark.sql(f'DESCRIBE {HIVE_DATABASE}.{tbl}').toPandas()
        cols = desc['col_name'].str.lower()
        region_cols = cols[cols.str.contains('region|address|city|terr|geo|place|location|addr', case=False, na=False)]
        if not region_cols.empty:
            tables_with_region.append((tbl, list(region_cols.values)))
    except Exception:
        pass

print(f'Таблицы с колонками, содержащими "region/address/city/geo":')
if tables_with_region:
    for tbl, cols in tables_with_region:
        print(f'  {tbl}: {cols}')
else:
    print('  Не найдено в основной БД.')

---
## 6. Поиск таблиц с сегментом / моделью / рейтингом

In [ ]:
tables_with_segment = []

for tbl in table_names.values:
    try:
        desc = spark.sql(f'DESCRIBE {HIVE_DATABASE}.{tbl}').toPandas()
        cols = desc['col_name'].str.lower()
        seg_cols = cols[cols.str.contains('segment|model|rating|score|categ|class|tier|grade', case=False, na=False)]
        if not seg_cols.empty:
            tables_with_segment.append((tbl, list(seg_cols.values)))
    except Exception:
        pass

print(f'Таблицы с колонками "segment/model/rating/score":')
if tables_with_segment:
    for tbl, cols in tables_with_segment:
        print(f'  {tbl}: {cols}')
else:
    print('  Не найдено в основной БД.')

---
## 7. Поиск таблиц со статусом клиента / проспекта

In [ ]:
tables_with_status = []

for tbl in table_names.values:
    try:
        desc = spark.sql(f'DESCRIBE {HIVE_DATABASE}.{tbl}').toPandas()
        cols = desc['col_name'].str.lower()
        status_cols = cols[cols.str.contains('status|state|stage|prospect|lead|attract|flag', case=False, na=False)]
        if not status_cols.empty:
            tables_with_status.append((tbl, list(status_cols.values)))
    except Exception:
        pass

print(f'Таблицы с колонками "status/stage/prospect/flag":')
if tables_with_status:
    for tbl, cols in tables_with_status:
        print(f'  {tbl}: {cols}')
else:
    print('  Не найдено.')

---
## 8. Поиск таблиц с выручкой / доходом / продуктами

In [ ]:
tables_with_revenue = []

for tbl in table_names.values:
    try:
        desc = spark.sql(f'DESCRIBE {HIVE_DATABASE}.{tbl}').toPandas()
        cols = desc['col_name'].str.lower()
        rev_cols = cols[cols.str.contains('revenue|turnover|income|profit|deal|product|contract|balance|amt|amount', case=False, na=False)]
        if not rev_cols.empty:
            tables_with_revenue.append((tbl, list(rev_cols.values)))
    except Exception:
        pass

print(f'Таблицы с колонками "revenue/turnover/income/deal/product/amount":')
if tables_with_revenue:
    for tbl, cols in tables_with_revenue:
        print(f'  {tbl}: {cols}')
else:
    print('  Не найдено.')

---
## 9. Детальный анализ найденных таблиц

После секций 4-8 у вас будет список таблиц-кандидатов.  
Впишите их имена ниже для детального анализа (схема + примеры данных).

In [ ]:
# =====================================================
# ВПИШИТЕ СЮДА НАЙДЕННЫЕ ТАБЛИЦЫ ДЛЯ ДЕТАЛЬНОГО АНАЛИЗА
# =====================================================

TABLES_TO_INSPECT = [
    # Пример:
    # 'clientactivity_sdim',
    # 'client_segment_shist',
    # 'client_address_shist',
]

In [ ]:
for tbl in TABLES_TO_INSPECT:
    # Схема
    describe_table_full(tbl)

    # Примеры данных
    print(f'\nПример данных ({tbl}):')
    try:
        spark.sql(f'SELECT * FROM {HIVE_DATABASE}.{tbl} LIMIT 10').show(truncate=50, vertical=True)
    except Exception as e:
        print(f'  ERROR: {e}')

    # Количество строк (может быть медленным для больших таблиц)
    try:
        cnt = spark.sql(f'SELECT COUNT(*) AS cnt FROM {HIVE_DATABASE}.{tbl}').collect()[0]['cnt']
        print(f'  Строк: {cnt:,}')
    except Exception as e:
        print(f'  COUNT failed: {e}')

---
## 10. Анализ справочника типов клиентов (`clienttype_ldim`)

Проверим: есть ли разделение на клиентов / проспектов через тип.

In [ ]:
describe_table_full('clienttype_ldim')

print('\nВсе значения clienttype_ldim:')
spark.sql(f'SELECT * FROM {HIVE_DATABASE}.clienttype_ldim').show(100, truncate=False)

In [ ]:
# Проверим clientstatus_ldim — есть ли статусы проспект/клиент
describe_table_full('clientstatus_ldim')

print('\nВсе значения clientstatus_ldim:')
spark.sql(f'SELECT * FROM {HIVE_DATABASE}.clientstatus_ldim').show(100, truncate=False)

---
## 11. Анализ `account_sdim` — что можно извлечь из счетов

Счета могут дать: регион через `salesplace_ccode` (точка продаж), продуктовую информацию.

In [ ]:
describe_table_full('account_sdim')

In [ ]:
# salesplace_ccode — может быть proxy для региона
print('Уникальные salesplace_ccode (топ-20 по частоте):')
spark.sql(f"""
    SELECT salesplace_ccode, COUNT(*) AS cnt
    FROM {HIVE_DATABASE}.account_sdim
    WHERE salesplace_ccode IS NOT NULL AND salesplace_ccode != ''
    GROUP BY salesplace_ccode
    ORDER BY cnt DESC
    LIMIT 20
""").show(truncate=False)

In [ ]:
# accountkind_uk — виды счетов (расчётный, депозитный, ...)
print('Уникальные accountkind_uk (типы счетов):')
spark.sql(f"""
    SELECT accountkind_uk, COUNT(*) AS cnt
    FROM {HIVE_DATABASE}.account_sdim
    WHERE accountkind_uk IS NOT NULL
    GROUP BY accountkind_uk
    ORDER BY cnt DESC
    LIMIT 20
""").show(truncate=False)

---
## 12. Анализ транзакций — что доступно для поведенческих фичей

Подтверждаем структуру `paymentcounteragent_stran` и проверяем категории денежных потоков.

In [ ]:
# cashflowcategory_dk — категории денежных потоков
# Может быть полезно для сегментации по типу бизнеса
print('Категории денежных потоков (cashflowcategory_dk):')
spark.sql(f"""
    SELECT cashflowcategory_dk, COUNT(*) AS cnt
    FROM {HIVE_DATABASE}.paymentcounteragent_stran
    WHERE date_part >= 20250101 AND date_part <= 20250131
      AND cashflowcategory_dk IS NOT NULL
    GROUP BY cashflowcategory_dk
    ORDER BY cnt DESC
    LIMIT 30
""").show(truncate=False)

In [ ]:
# Пример транзакции со всеми полями
print('Пример транзакции (все поля):')
spark.sql(f"""
    SELECT *
    FROM {HIVE_DATABASE}.paymentcounteragent_stran
    WHERE date_part >= 20250101 AND date_part <= 20250131
    LIMIT 3
""").show(truncate=50, vertical=True)

---
## 13. Производные данные: что можно рассчитать из транзакций

Даже без прямых полей ОКВЭД/выручки/сегмента, из транзакций можно извлечь суррогатные признаки для look-alike.

Проверим, достаточно ли данных для вычисления агрегатов по месяцам.

In [ ]:
# Проверяем диапазон дат и объём данных по месяцам
print('Объём транзакций по месяцам (2025):')
spark.sql(f"""
    SELECT
        CAST(FLOOR(date_part / 100) AS INT) AS year_month,
        COUNT(*) AS tx_count,
        COUNT(DISTINCT client_uk) AS unique_clients,
        SUM(rur_amt) AS total_amount
    FROM {HIVE_DATABASE}.paymentcounteragent_stran
    WHERE date_part >= 20250101 AND date_part <= 20251231
      AND (deleted_flag IS NULL OR deleted_flag != 'Y')
    GROUP BY CAST(FLOOR(date_part / 100) AS INT)
    ORDER BY year_month
""").show(20, truncate=False)

In [ ]:
# Распределение клиентов по числу транзакций за 2025
print('Распределение клиентов по активности (2025):')
spark.sql(f"""
    SELECT
        CASE
            WHEN tx_cnt <= 5    THEN '01: 1-5'
            WHEN tx_cnt <= 20   THEN '02: 6-20'
            WHEN tx_cnt <= 100  THEN '03: 21-100'
            WHEN tx_cnt <= 500  THEN '04: 101-500'
            WHEN tx_cnt <= 1000 THEN '05: 501-1000'
            ELSE '06: 1000+'
        END AS activity_band,
        COUNT(*) AS client_count
    FROM (
        SELECT client_uk, COUNT(*) AS tx_cnt
        FROM {HIVE_DATABASE}.paymentcounteragent_stran
        WHERE date_part >= 20250101 AND date_part <= 20251231
          AND (deleted_flag IS NULL OR deleted_flag != 'Y')
        GROUP BY client_uk
    ) t
    GROUP BY
        CASE
            WHEN tx_cnt <= 5    THEN '01: 1-5'
            WHEN tx_cnt <= 20   THEN '02: 6-20'
            WHEN tx_cnt <= 100  THEN '03: 21-100'
            WHEN tx_cnt <= 500  THEN '04: 101-500'
            WHEN tx_cnt <= 1000 THEN '05: 501-1000'
            ELSE '06: 1000+'
        END
    ORDER BY activity_band
""").show(truncate=False)

---
## 14. Поиск таблиц с продуктами / сделками / каналами

Продуктовый профиль клиента (какие продукты использует) может быть суррогатом для сегмента.

In [ ]:
tables_with_products = []

for tbl in table_names.values:
    try:
        desc = spark.sql(f'DESCRIBE {HIVE_DATABASE}.{tbl}').toPandas()
        cols = desc['col_name'].str.lower()
        prod_cols = cols[cols.str.contains('product|deal|contract|service|offer|tariff|package', case=False, na=False)]
        if not prod_cols.empty:
            tables_with_products.append((tbl, list(prod_cols.values)))
    except Exception:
        pass

print(f'Таблицы с колонками "product/deal/contract/service":')
if tables_with_products:
    for tbl, cols in tables_with_products:
        print(f'  {tbl}: {cols}')
else:
    print('  Не найдено.')

---
## 15. Сводная карта данных

Заполните таблицу по результатам разведки:

In [ ]:
# =====================================================
# ЗАПОЛНИТЕ ПО РЕЗУЛЬТАТАМ ЗАПУСКА
# =====================================================

data_map = pd.DataFrame([
    {'Данные': 'ОКВЭД',           'Таблица': '???', 'Колонка': '???',          'Комментарий': 'Не в client_sdim'},
    {'Данные': 'Регион',           'Таблица': '???', 'Колонка': '???',          'Комментарий': 'Может быть salesplace_ccode?'},
    {'Данные': 'Выручка',          'Таблица': '???', 'Колонка': '???',          'Комментарий': 'Или SUM(rur_amt)?'},
    {'Данные': 'Модельный сегмент','Таблица': '???', 'Колонка': '???',          'Комментарий': ''},
    {'Данные': 'Статус (клиент/проспект)', 'Таблица': '???', 'Колонка': '???',  'Комментарий': ''},
    {'Данные': 'Тип клиента',      'Таблица': 'clienttype_ldim', 'Колонка': 'NAME', 'Комментарий': 'ЮЛ/ФЛ/ИП'},
    {'Данные': 'Транзакции',       'Таблица': 'paymentcounteragent_stran', 'Колонка': 'rur_amt, income_flag, date_part', 'Комментарий': 'OK'},
    {'Данные': 'Контрагенты',      'Таблица': 'paymentcounteragent_stran', 'Колонка': 'client_contr_uk', 'Комментарий': 'OK'},
    {'Данные': 'ИНН',              'Таблица': 'account_sdim', 'Колонка': 'client_taxpayer_ccode', 'Комментарий': 'OK'},
    {'Данные': 'Зарплатные связи', 'Таблица': 'clnt2dealsalary_shist', 'Колонка': 'client_uk, dealsalary_uk', 'Комментарий': 'OK'},
])

print('='*80)
print('КАРТА ДАННЫХ ДЛЯ LOOK-ALIKE & BEHAVIORAL SEGMENTATION')
print('='*80)
display(data_map)

---
## 16. Следующие шаги

По результатам разведки определите:

1. **Есть прямые источники** (ОКВЭД, регион, сегмент в БД) → используем как есть
2. **Нет прямых, но есть суррогаты**:
   - Регион ← `salesplace_ccode` из `account_sdim`
   - ОКВЭД ← по ИНН через внешний справочник (если доступен)
   - Выручка ← `SUM(rur_amt)` из `paymentcounteragent_stran` (оборот по счёту ≈ proxy выручки)
   - Сегмент ← вычисляем через кластеризацию (именно это задача проекта)
   - Статус ← наличие/отсутствие транзакций, или `deleted_flag` / `end_date` в `client_sdim`
3. **Данных нет совсем** → нужна выгрузка из CRM или СПАРК

Обновите ячейку 15 (карту данных) и передайте результат для проектирования ETL.

---

## Глоссарий терминов

| Термин | Описание |
|--------|----------|
| **ОКВЭД** | Общероссийский классификатор видов экономической деятельности; двузначный код отрасли (напр. 46 = оптовая торговля) |
| **Модельный сегмент** | Категория клиента по размеру/типу бизнеса (напр. Micro, SME, Mid-Corporate, Large), обычно рассчитывается CRM-моделью |
| **Проспект** | Потенциальный клиент, ещё не ставший клиентом банка |
| **Look-Alike** | Модель поиска компаний, похожих на эталонную группу (лучших клиентов) по набору признаков |
| **salesplace_ccode** | Код точки продаж (отделение банка) — может служить суррогатом региона |
| **cashflowcategory_dk** | Категория денежного потока в транзакции — может характеризовать тип бизнес-операции |
| **client_sdim** | Справочник клиентов (182M строк, 65 полей) — основная таблица с данными о клиентах |
| **paymentcounteragent_stran** | Таблица транзакций (4.98B строк) — основной источник поведенческих данных |

---

**Следующий шаг**: По результатам разведки — спроектировать ETL для извлечения данных (`01_data_extraction.ipynb`).